## Metrics

汇总常见2分类的指标，例如: AUC，ROC曲线，ACC, 敏感性， 特异性，精确度，召回率，PPV, NPV, F1

具体的介绍，可以参考一下：https://blog.csdn.net/sunflower_sara/article/details/81214897

# 选择你需要的结果

In [ ]:
import pandas as pd
import numpy  as np
import re
import os
from onekey_algo.custom.components import metrics
from onekey_algo.custom.components.comp1 import draw_roc, normalize_df
from matplotlib import pyplot as plt
from onekey_algo.custom.utils import print_join_info
from onekey_algo import get_param_in_cwd
from datetime import datetime

task_type = 'DeepLearning'

pd.set_option('display.max_rows', None)
os.makedirs('img', exist_ok=True)
os.makedirs('results', exist_ok=True)
subsets = get_param_in_cwd('subsets')
label_file = get_param_in_cwd('label_file')
model_root = get_param_in_cwd('model_root')
group_info = pd.read_csv(label_file)
def get_log(log_path, map2gz:bool = True):
    log_ = pd.read_csv(log_path, names=['fname', 'pred_score', 'pred_label', 'gt'], sep='\t')
    if map2gz:
        log_['ID'] = log_['fname'].map(lambda x: os.path.splitext(os.path.basename(x))[0] + '.gz')
    else:
        log_['ID'] = log_['fname'].map(lambda x: os.path.basename(x))
    return log_

def map_mn(x):
    return x.replace('densen', 'DenseN').replace('resnet', 'ResNet').replace('vgg', 'VGG').replace('inception_v3', 'InceptionV3')

all_log_ = []
metrics_dfs = []
for data_pattern in [get_param_in_cwd('data_pattern')]:
    for epoch_ in [get_param_in_cwd('sel_epoch')]:
        metric_results = []
        all_preds = []
        all_gts = []
        all_model_names = []
        for model in [get_param_in_cwd('sel_model_name') or get_param_in_cwd('model_names')[0]]:
            all_pred = []
            all_gt = []
            all_groups = []
            # 评估的事viz的最佳非训练集表现的模型
            if epoch_ is not None and epoch_ != '':
            # 指定epoch评估的模型
                val_log = pd.concat([get_log(os.path.join(model_root, model, f"train/Epoch-{epoch_}.txt")),
                                     get_log(os.path.join(model_root, model, f"valid/Epoch-{epoch_}.txt"))], axis=0)
            else:
                val_log = pd.concat([get_log(os.path.join(model_root, model, f"viz/BST_TRAIN_RESULTS.txt")),
                                     get_log(os.path.join(model_root, model, f"viz/BST_VAL_RESULTS.txt"))], axis=0)
            val_log = pd.merge(val_log.drop_duplicates('ID'), group_info, on='ID', how='inner')
            val_log['gt'] = val_log['label']
            val_log['model'] = f"{model}"
            ug_groups = subsets
            ul_labels = np.unique(val_log['pred_label'])
            for g in ug_groups:
                sub_group = val_log[val_log['group'] == g]
                sub_group['label-1'] = list(map(lambda x: x[0] if x[1] == 1 else 1-x[0], 
                                                np.array(sub_group[['pred_score', 'pred_label']])))
                sub_group['label-0'] = 1 - sub_group['label-1']
                sub_group[['ID', 'label-0', 'label-1']].to_csv(os.path.join('results', f'{task_type}_{model}_{g}.csv'), index=False)
                all_groups.append(g)                    
                all_log_.append(sub_group)
                for ul in [1]:
                    pred_score = np.array(sub_group['label-1'])
                    gt = [1 if gt_ == ul else 0 for gt_ in np.array(sub_group['gt'])]
                    acc, auc, ci, tpr, tnr, ppv, npv, p, r, f1, thres = metrics.analysis_pred_binary(gt, pred_score, use_youden=True)
                    ci = f"{ci[0]:.4f}-{ci[1]:.4f}"
                    metric_results.append([model, acc, auc, ci, tpr, tnr, ppv, npv, p, r, f1, thres, g])
                    all_pred.append(pred_score)
                    all_gt.append(gt)
            # 绘制每个模型的ROC
            draw_roc(all_gt, all_pred, labels=all_groups, title=f"Model: {map_mn(model)}")
            plt.savefig(f'img/{task_type}_{model}_roc.svg', bbox_inches='tight')
            plt.show()
            # 整合到所有模型汇总。
            all_preds.extend(all_pred)
            all_gts.extend(all_gt)
            all_model_names.append(model)
        # for gi, g in enumerate(all_groups):
        #     draw_roc(all_gts[gi::len(all_groups)], all_preds[gi::len(all_groups)], 
        #              labels=[map_mn(m) for m in all_model_names], 
        #              title=f"Cohort {g}")
        #     plt.savefig(f'img/DL_{g}_roc.svg', bbox_inches='tight')
        #     plt.show()
        metrics_df = pd.DataFrame(metric_results, 
                                  columns=['ModelName', 'Acc', 'AUC', '95% CI', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 
                                           'Precision', 'Recall', 'F1',
                                           'Youden', 'Cohort'])
        metrics_df['Epoch'] = epoch_
        display(metrics_df)
        metrics_dfs.append(metrics_df)
pd.concat(metrics_dfs, axis=0)